# Financial Analysis Five-Table Generator

This notebook builds five modular financial label tables for the selected 100,000 UK companies.

- Company universe: `UKcompanies_active_account_category_sample_100k.csv`
- Financial source: all monthly Companies House Accounts Bulk ZIPs from July 2024 to June 2026
- Quantile labels: 30th/70th percentiles within `primary_sector + Accounts_AccountCategory`
- Fallback: sector-level, then global thresholds when a group has fewer than 30 eligible observations or tied thresholds
- Current snapshot: latest account period, not the historical best-evidence period

The raw company list and Accounts ZIPs are never modified. The five outputs remain separate so that current-state labels, longitudinal targets, and data-quality controls are not mixed accidentally.


In [1]:
from pathlib import Path
from IPython.display import display
import calendar
import json
import os
import re
import time
import zipfile

import numpy as np
import pandas as pd

COMPANY_CSV = Path(r"E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k.csv")
DISTRIBUTION_CSV = Path(r"E:\000硕士毕设\公司选取\UKcompanies_active_account_category_sample_100k_distribution.csv")
ACCOUNTS_ZIP_DIR = Path(r"E:\000硕士毕设\财务数据\Local Large Data\Accounts Data_2024.7_2026.6")

# This cache was previously parsed from the same 100k company set and the same 24 ZIP files.
# Set REBUILD_FROM_RAW=True to ignore it and reparse every matched account directly from the ZIPs.
PARSED_FILING_CACHE = Path(r"E:\000硕士毕设\公司+财务数据匹配\financial_features_filing_100k.csv")
REBUILD_FROM_RAW = os.getenv("FINANCIAL_FIVE_TABLES_REBUILD", "false").lower() == "true"

DEFAULT_OUTPUT_DIR = r"E:\000硕士毕设\财务数据分析的五大表"
OUTPUT_DIR = Path(os.getenv("FINANCIAL_FIVE_TABLES_OUTPUT_DIR", DEFAULT_OUTPUT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = pd.Timestamp("2026-06-30")
LOW_Q = 0.30
HIGH_Q = 0.70
MIN_GROUP_N = 30
ANNUAL_GAP_MIN_DAYS = 250
ANNUAL_GAP_MAX_DAYS = 550
CSV_ENCODING = "utf-8-sig"

OUTPUT_FILES = {
    "status": OUTPUT_DIR / "01_financial_status_labels_100k.csv",
    "scale": OUTPUT_DIR / "02_financial_scale_labels_100k.csv",
    "change": OUTPUT_DIR / "03_financial_change_labels.csv",
    "transition": OUTPUT_DIR / "04_financial_transition_labels.csv",
    "quality": OUTPUT_DIR / "05_financial_data_quality_labels_100k.csv",
    "guide_cn": OUTPUT_DIR / "financial_five_tables_guide_CN.md",
    "guide_en": OUTPUT_DIR / "financial_five_tables_guide_EN.md",
}

print("Output directory:", OUTPUT_DIR)
print("Rebuild raw Accounts ZIPs:", REBUILD_FROM_RAW)


Output directory: E:\000硕士毕设\财务数据分析的五大表
Rebuild raw Accounts ZIPs: False


## 1. Load and validate the 100k company universe

The distribution file is used as a control total. Company numbers are normalized before any matching.


In [2]:
def normalise_company_number(value):
    if pd.isna(value):
        return ""
    s = re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())
    if not s:
        return ""
    if s.isdigit():
        return s.zfill(8)
    if len(s) < 8 and re.match(r"^[A-Z]{2}\d+$", s):
        return s[:2] + s[2:].zfill(6)
    return s


companies = pd.read_csv(COMPANY_CSV, low_memory=False)
distribution_reference = pd.read_csv(DISTRIBUTION_CSV, low_memory=False)
companies["CompanyNumber_norm"] = companies["CompanyNumber"].map(normalise_company_number)

assert len(companies) == 100_000, f"Expected 100,000 rows, found {len(companies):,}"
assert companies["CompanyNumber_norm"].nunique() == 100_000, "CompanyNumber is not unique after normalization"
assert companies["CompanyNumber_norm"].ne("").all(), "Blank normalized company number found"

observed_category = companies["Accounts_AccountCategory"].value_counts().sort_index()
expected_category = distribution_reference.set_index("Accounts_AccountCategory")["sample_count"].sort_index()
assert observed_category.equals(expected_category), "Company account-category distribution differs from the control file"

company_lookup_cols = [
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "CompanyCategory", "company_age_years", "primary_sic_code",
]
company_lookup = companies[company_lookup_cols].copy()

print("Validated companies:", f"{len(companies):,}")
display(distribution_reference)


Validated companies: 100,000


,Accounts_AccountCategory,eligible_count,eligible_percent,sample_quota,sample_percent,sample_count,sample_percent_actual,quota_minus_actual
0,MICRO ENTITY,1079219,0.532408,53241,53.24%,53241,0.53241,0
1,TOTAL EXEMPTION FULL,748335,0.369174,36917,36.92%,36917,0.36917,0
2,UNAUDITED ABRIDGED,91523,0.045151,4515,4.52%,4515,0.04515,0
3,FULL,37182,0.018343,1834,1.83%,1834,0.01834,0
4,SMALL,36109,0.017814,1781,1.78%,1781,0.01781,0
5,AUDIT EXEMPTION SUBSIDIARY,18274,0.009015,902,0.90%,902,0.00902,0
6,GROUP,12698,0.006264,627,0.63%,627,0.00627,0
7,MEDIUM,3713,0.001832,183,0.18%,183,0.00183,0


## 2. Parse or reuse filing-level financial data

The cache is reused only when its company set and source ZIP set match the current inputs exactly. Otherwise, the notebook rebuilds from the raw ZIPs. Set the environment variable `FINANCIAL_FIVE_TABLES_REBUILD=true` to force a full rebuild.


In [3]:
def parse_member_name(member_name):
    name = Path(member_name).name
    m = re.search(r"[_-]([A-Z]{0,2}\d{6,8})[_-](\d{8})\.(html|xml|zip)$", name, flags=re.IGNORECASE)
    if not m:
        return None
    company_number = normalise_company_number(m.group(1))
    raw_date = m.group(2)
    period_end = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}"
    return company_number, period_end, m.group(3).lower()


def attrs_to_dict(attr_text):
    return {
        key.lower(): value
        for key, value in re.findall(r"([\w:.-]+)\s*=\s*['\"]([^'\"]*)['\"]", attr_text or "")
    }


def strip_tags(text):
    text = re.sub(r"<[^>]+>", "", text or "")
    return text.replace("&nbsp;", " ").replace("&#160;", " ").replace("&amp;", "&").strip()


def parse_numeric(raw_value, scale=None, sign=None):
    if raw_value is None:
        return np.nan
    s = re.sub(r"\s+", "", strip_tags(str(raw_value)).replace(",", "").replace("£", "").replace("$", ""))
    if s in {"", "-", "—", "nan", "None"}:
        return np.nan
    negative_by_parentheses = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    try:
        value = float(s)
    except Exception:
        return np.nan
    if negative_by_parentheses:
        value = -value
    if sign == "-":
        value = -value
    try:
        if scale not in {None, ""}:
            value *= 10 ** int(scale)
    except Exception:
        pass
    return value


def extract_context_info(text):
    context_info = {}
    pattern = r"<(?:\w+:)?context\b([^>]*)>(.*?)</(?:\w+:)?context>"
    for m in re.finditer(pattern, text, flags=re.IGNORECASE | re.DOTALL):
        attrs = attrs_to_dict(m.group(1))
        context_id = attrs.get("id")
        if not context_id:
            continue
        body = m.group(2)
        end_match = re.search(r"<(?:\w+:)?endDate>(.*?)</(?:\w+:)?endDate>", body, flags=re.IGNORECASE | re.DOTALL)
        instant_match = re.search(r"<(?:\w+:)?instant>(.*?)</(?:\w+:)?instant>", body, flags=re.IGNORECASE | re.DOTALL)
        end_date = strip_tags(end_match.group(1)) if end_match else (strip_tags(instant_match.group(1)) if instant_match else "")
        dimension_count = len(re.findall(r"(?:explicitMember|typedMember)", body, flags=re.IGNORECASE))
        context_info[context_id] = {"context_end_date": end_date, "context_dimension_count": dimension_count}
    return context_info


METRIC_TAG_PATTERNS = {
    "turnover": [r"TurnoverRevenue$", r"^Turnover$", r"RevenueFromSaleOfGoods$", r"RevenueFromRenderingServices$", r"RevenueFromContractsWithCustomers$", r"GrossOperatingRevenue$"],
    "current_assets": [r"CurrentAssets$"],
    "fixed_assets": [r"FixedAssets$"],
    "net_current_assets_liabilities": [r"NetCurrentAssetsLiabilities$"],
    "total_assets_less_current_liabilities": [r"TotalAssetsLessCurrentLiabilities$"],
    "net_assets_liabilities": [r"NetAssetsLiabilities$", r"NetAssets$"],
    "equity": [r"Equity$", r"ShareholderFunds$", r"CapitalAndReserves$"],
    "cash": [r"CashBankOnHand$", r"CashAndCashEquivalents$", r"CashAtBankAndInHand$"],
    "debtors": [r"Debtors$", r"DebtorsAmountsFallingDueWithinOneYear$"],
    "creditors_within_one_year": [r"CreditorsAmountsFallingDueWithinOneYear$", r"CreditorsDueWithinOneYear$"],
    "creditors_after_one_year": [r"CreditorsAmountsFallingDueAfterMoreThanOneYear$", r"CreditorsDueAfterOneYear$", r"CreditorsAmountsFallingDueAfterOneYear$"],
    "creditors_total": [r"Creditors$", r"TotalCreditors$"],
    "employees": [r"AverageNumberEmployeesDuringPeriod$", r"AverageNumberOfEmployeesDuringPeriod$", r"EmployeesTotal$"],
    "profit_loss": [r"ProfitLoss$", r"ProfitLossBeforeTax$", r"OperatingProfitLoss$"],
}
EXCLUDE_FACT_NAME_PATTERNS = [
    r"Policy", r"Description", r"Disclosure", r"Narrative", r"TextBlock", r"Taxonomy",
    r"RevenueRecognition", r"DeferredTax", r"IncomeTax", r"TaxCredit", r"CostOfSales",
]
CORE_PROXY_FIELDS = ["current_assets", "net_assets_liabilities", "equity", "creditors_total", "employees", "cash", "debtors"]


def metric_for_fact_name(fact_name):
    local = fact_name.split(":")[-1]
    if any(re.search(p, local, flags=re.IGNORECASE) for p in EXCLUDE_FACT_NAME_PATTERNS):
        return None
    for metric, patterns in METRIC_TAG_PATTERNS.items():
        if any(re.search(pattern, local, flags=re.IGNORECASE) for pattern in patterns):
            return metric
    return None


def selection_score(context_end_date, period_end, dimension_count):
    score = 20 if context_end_date and period_end and str(context_end_date)[:10] != str(period_end)[:10] else 0
    return score + int(dimension_count or 0) * 5


def parse_financial_facts(text, company_number, period_end, source_zip, internal_filename):
    context_info = extract_context_info(text)
    facts = []
    ix_pattern = re.compile(r"<ix:(?:nonFraction|nonNumeric)\b([^>]*)>(.*?)</ix:(?:nonFraction|nonNumeric)>", re.IGNORECASE | re.DOTALL)
    for m in ix_pattern.finditer(text):
        attrs = attrs_to_dict(m.group(1))
        fact_name = attrs.get("name", "")
        metric = metric_for_fact_name(fact_name)
        if not metric:
            continue
        context_ref = attrs.get("contextref", "")
        ctx = context_info.get(context_ref, {})
        value = parse_numeric(m.group(2), scale=attrs.get("scale"), sign=attrs.get("sign"))
        if pd.notna(value):
            facts.append((metric, value, selection_score(ctx.get("context_end_date", ""), period_end, ctx.get("context_dimension_count", 0))))

    xml_pattern = re.compile(r"<([A-Za-z0-9_:\.-]+)\b([^>]*)>([^<>{}]{1,200})</\1>", re.IGNORECASE | re.DOTALL)
    for m in xml_pattern.finditer(text):
        fact_name = m.group(1)
        metric = metric_for_fact_name(fact_name)
        if not metric:
            continue
        attrs = attrs_to_dict(m.group(2))
        context_ref = attrs.get("contextref", "")
        ctx = context_info.get(context_ref, {})
        value = parse_numeric(m.group(3), scale=attrs.get("scale"), sign=attrs.get("sign"))
        if pd.notna(value):
            facts.append((metric, value, selection_score(ctx.get("context_end_date", ""), period_end, ctx.get("context_dimension_count", 0))))
    return facts


def select_metric_values(facts):
    if not facts:
        return {}
    facts_df = pd.DataFrame(facts, columns=["metric", "numeric_value", "selection_score"])
    selected = {
        metric: float(group.sort_values("selection_score").iloc[0]["numeric_value"])
        for metric, group in facts_df.groupby("metric", sort=False)
    }
    if "creditors_total" not in selected:
        within = selected.get("creditors_within_one_year")
        after = selected.get("creditors_after_one_year")
        if within is not None or after is not None:
            selected["creditors_total"] = float((within or 0) + (after or 0))
    return selected


def assign_evidence_tier(row):
    if pd.notna(row.get("turnover")):
        return "T1_observed_turnover"
    core_count = int(row.get("available_core_proxy_field_count", 0) or 0)
    if core_count >= 4:
        return "T2_balance_sheet_rich"
    if core_count >= 2:
        return "T3_balance_sheet_partial"
    return "T4_account_category_only"


def safe_read_text_from_zip(zf, entry):
    raw = zf.read(entry)
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return raw.decode(encoding)
        except Exception:
            pass
    return raw.decode("latin-1", errors="replace")


def rebuild_filing_features_from_raw():
    target_ids = set(companies["CompanyNumber_norm"])
    zip_paths = sorted(ACCOUNTS_ZIP_DIR.glob("*.zip"))
    if len(zip_paths) != 24:
        raise ValueError(f"Expected 24 monthly ZIPs, found {len(zip_paths)}")

    match_rows = []
    for zip_idx, zip_path in enumerate(zip_paths, start=1):
        with zipfile.ZipFile(zip_path) as zf:
            for entry in zf.infolist():
                if entry.is_dir():
                    continue
                parsed = parse_member_name(entry.filename)
                if parsed and parsed[0] in target_ids:
                    company_number, period_end, file_format = parsed
                    match_rows.append({
                        "CompanyNumber_norm": company_number, "period_end": period_end,
                        "source_zip": zip_path.name, "internal_filename": entry.filename,
                        "file_format": file_format, "file_size": entry.file_size,
                    })
        print(f"Indexed ZIP {zip_idx}/24: {zip_path.name}")

    match_index = pd.DataFrame(match_rows)
    feature_rows, error_rows = [], []
    for zip_idx, (zip_name, rows) in enumerate(match_index.groupby("source_zip", sort=False), start=1):
        with zipfile.ZipFile(ACCOUNTS_ZIP_DIR / zip_name) as zf:
            for row in rows.itertuples(index=False):
                try:
                    text = safe_read_text_from_zip(zf, row.internal_filename)
                    facts = parse_financial_facts(text, row.CompanyNumber_norm, row.period_end, row.source_zip, row.internal_filename)
                    selected = select_metric_values(facts)
                    result = {
                        "CompanyNumber_norm": row.CompanyNumber_norm, "period_end": row.period_end,
                        "source_zip": row.source_zip, "internal_filename": row.internal_filename,
                        "file_format": row.file_format, "file_size": row.file_size,
                        "parsed_ok": True, "facts_extracted_count": len(facts), **selected,
                    }
                    result["available_core_proxy_field_count"] = sum(pd.notna(result.get(c)) for c in CORE_PROXY_FIELDS)
                    any_fields = set(CORE_PROXY_FIELDS + ["turnover", "fixed_assets", "profit_loss", "creditors_within_one_year", "creditors_after_one_year"])
                    result["available_any_financial_field_count"] = sum(pd.notna(result.get(c)) for c in any_fields)
                    result["financial_evidence_tier"] = assign_evidence_tier(result)
                    feature_rows.append(result)
                except Exception as exc:
                    error_rows.append({"CompanyNumber_norm": row.CompanyNumber_norm, "period_end": row.period_end, "source_zip": zip_name, "error": repr(exc)})
        print(f"Parsed ZIP {zip_idx}/24: {zip_name}; cumulative matched filings={len(feature_rows):,}")

    if error_rows:
        pd.DataFrame(error_rows).to_csv(OUTPUT_DIR / "accounts_parse_errors_100k.csv", index=False, encoding=CSV_ENCODING)
    return pd.DataFrame(feature_rows)


In [4]:
zip_names = {path.name for path in ACCOUNTS_ZIP_DIR.glob("*.zip")}
cache_valid = False

if PARSED_FILING_CACHE.exists() and not REBUILD_FROM_RAW:
    cache_source = pd.read_csv(PARSED_FILING_CACHE, usecols=["CompanyNumber_norm", "source_zip"], dtype="string")
    cached_ids = set(cache_source["CompanyNumber_norm"].map(normalise_company_number))
    cached_zip_names = set(cache_source["source_zip"].dropna())
    cache_valid = cached_ids.issubset(set(companies["CompanyNumber_norm"])) and cached_zip_names == zip_names and len(zip_names) == 24
    del cache_source

if cache_valid:
    filing_features = pd.read_csv(PARSED_FILING_CACHE, low_memory=False)
    print("Using validated filing cache:", PARSED_FILING_CACHE)
else:
    filing_features = rebuild_filing_features_from_raw()
    print("Rebuilt filing features from raw Accounts ZIPs")

filing_features["CompanyNumber_norm"] = filing_features["CompanyNumber_norm"].map(normalise_company_number)
filing_features = filing_features[filing_features["CompanyNumber_norm"].isin(set(companies["CompanyNumber_norm"]))].copy()

assert len(zip_names) == 24, f"Expected 24 monthly ZIPs, found {len(zip_names)}"
assert set(filing_features["source_zip"].dropna()) == zip_names, "Filing data does not cover exactly the 24 source ZIPs"

print("Filing rows:", f"{len(filing_features):,}")
print("Matched companies:", f"{filing_features['CompanyNumber_norm'].nunique():,}")


ValueError: Expected 24 monthly ZIPs, found 0

## 3. Build a deduplicated company-period panel

Duplicate filings for the same company and account period are resolved by taking the strongest evidence tier and then the latest available monthly batch. `available_date` is the month-end of the source Accounts Bulk ZIP.


In [ ]:
MONTH_TO_NUMBER = {month.lower(): number for number, month in enumerate(calendar.month_name) if month}


def source_zip_available_date(zip_name):
    match = re.search(r"Data-([A-Za-z]+)(\d{4})\.zip$", str(zip_name))
    if not match:
        return pd.NaT
    month = MONTH_TO_NUMBER.get(match.group(1).lower())
    if not month:
        return pd.NaT
    return pd.Timestamp(int(match.group(2)), month, 1) + pd.offsets.MonthEnd(0)


EVIDENCE_RANK = {
    "T1_observed_turnover": 1,
    "T2_balance_sheet_rich": 2,
    "T3_balance_sheet_partial": 3,
    "T4_account_category_only": 4,
}
FINANCIAL_METRICS = [
    "turnover", "cash", "creditors_total", "current_assets", "debtors", "employees",
    "equity", "fixed_assets", "net_assets_liabilities", "net_current_assets_liabilities",
    "profit_loss", "total_assets_less_current_liabilities",
]

for column in FINANCIAL_METRICS:
    if column not in filing_features:
        filing_features[column] = np.nan
    filing_features[column] = pd.to_numeric(filing_features[column], errors="coerce")

filing_features["period_end"] = pd.to_datetime(filing_features["period_end"], errors="coerce")
filing_features["available_date"] = filing_features["source_zip"].map(source_zip_available_date)
filing_features["evidence_rank"] = filing_features["financial_evidence_tier"].map(EVIDENCE_RANK).fillna(99)

panel = (
    filing_features
    .sort_values(
        ["CompanyNumber_norm", "period_end", "evidence_rank", "available_date"],
        ascending=[True, True, True, False],
    )
    .groupby(["CompanyNumber_norm", "period_end"], as_index=False)
    .head(1)
    .sort_values(["CompanyNumber_norm", "period_end"])
    .reset_index(drop=True)
)
panel = panel.merge(company_lookup, on="CompanyNumber_norm", how="left", validate="many_to_one")

latest = (
    panel.sort_values(["CompanyNumber_norm", "period_end", "evidence_rank", "available_date"], ascending=[True, False, True, False])
    .groupby("CompanyNumber_norm", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
latest_all = company_lookup.merge(latest.drop(columns=[c for c in company_lookup.columns if c != "CompanyNumber_norm"]), on="CompanyNumber_norm", how="left", validate="one_to_one")

print("Deduplicated company-period rows:", f"{len(panel):,}")
print("Latest matched companies:", f"{latest['CompanyNumber_norm'].nunique():,}")
display(panel.head())


Deduplicated company-period rows: 176,405
Latest matched companies: 94,312


,CompanyNumber_norm,period_end,source_zip,internal_filename,file_format,file_size,parsed_ok,facts_extracted_count,cash,creditors_total,...,financial_evidence_tier,turnover,available_date,evidence_rank,CompanyName,primary_sector,Accounts_AccountCategory,CompanyCategory,company_age_years,primary_sic_code
0,00031641,2024-05-31,Accounts_Monthly_Data-February2025.zip,Prod224_3600_00031641_20240531.html,html,82779,True,42,62524.0,51454.0,...,T2_balance_sheet_rich,NaN,2025-02-28,2,BARNSLEY ARCADE COMPANY LIMITED(THE),"Technology, legal & professional",SMALL,Private Limited Company,136.0,41100
1,00031641,2025-05-31,Accounts_Monthly_Data-January2026.zip,Prod224_2601_00031641_20250531.html,html,78037,True,32,47563.0,56157.0,...,T2_balance_sheet_rich,NaN,2026-01-31,2,BARNSLEY ARCADE COMPANY LIMITED(THE),"Technology, legal & professional",SMALL,Private Limited Company,136.0,41100
2,00038191,2024-03-31,Accounts_Monthly_Data-August2024.zip,Prod224_2466_00038191_20240331.html,html,344626,True,38,461714.0,6325.0,...,T2_balance_sheet_rich,NaN,2024-08-31,2,WHARFEDALE FARMERS'AUCTION MART LIMITED,Agriculture,SMALL,Private Limited Company,133.3,1629
3,00038191,2025-03-31,Accounts_Monthly_Data-August2025.zip,Prod224_2508_00038191_20250331.html,html,342343,True,38,547259.0,6670.0,...,T2_balance_sheet_rich,NaN,2025-08-31,2,WHARFEDALE FARMERS'AUCTION MART LIMITED,Agriculture,SMALL,Private Limited Company,133.3,1629
4,00041365,2023-12-31,Accounts_Monthly_Data-September2024.zip,Prod224_2476_00041365_20231231.html,html,117245,True,44,3244.0,50462.0,...,T2_balance_sheet_rich,NaN,2024-09-30,2,EYLAND & SONS LIMITED,Manufacturing,AUDIT EXEMPTION SUBSIDIARY,Private Limited Company,132.0,25930


## 4. Reusable 30/70 quantile labelling

Threshold hierarchy:

1. `primary_sector + Accounts_AccountCategory`, if at least 30 eligible observations and the two thresholds differ;
2. `primary_sector`;
3. all eligible companies.

Every label records the selected threshold scope and the numerical thresholds used.


In [ ]:
def signed_log1p(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return np.sign(numeric) * np.log1p(np.abs(numeric))


def add_quantile_band(df, value_col, prefix, eligibility=None):
    result = df.copy()
    values = pd.to_numeric(result[value_col], errors="coerce")
    eligible = values.notna() if eligibility is None else (values.notna() & pd.Series(eligibility, index=result.index).fillna(False))
    work = result.loc[eligible, ["primary_sector", "Accounts_AccountCategory"]].copy()
    work["_value"] = values.loc[eligible]

    group_stats = work.groupby(["primary_sector", "Accounts_AccountCategory"])["_value"].agg(
        group_n="count", group_low=lambda x: x.quantile(LOW_Q), group_high=lambda x: x.quantile(HIGH_Q)
    ).reset_index()
    sector_stats = work.groupby("primary_sector")["_value"].agg(
        sector_n="count", sector_low=lambda x: x.quantile(LOW_Q), sector_high=lambda x: x.quantile(HIGH_Q)
    ).reset_index()
    global_n = int(work["_value"].count())
    global_low = work["_value"].quantile(LOW_Q) if global_n else np.nan
    global_high = work["_value"].quantile(HIGH_Q) if global_n else np.nan

    temp = result[["primary_sector", "Accounts_AccountCategory"]].merge(group_stats, on=["primary_sector", "Accounts_AccountCategory"], how="left")
    temp = temp.merge(sector_stats, on="primary_sector", how="left")
    use_group = (temp["group_n"] >= MIN_GROUP_N) & (temp["group_low"] < temp["group_high"])
    use_sector = (~use_group) & (temp["sector_n"] >= MIN_GROUP_N) & (temp["sector_low"] < temp["sector_high"])

    low = np.select([use_group, use_sector], [temp["group_low"], temp["sector_low"]], default=global_low)
    high = np.select([use_group, use_sector], [temp["group_high"], temp["sector_high"]], default=global_high)
    scope = np.select([use_group, use_sector], ["sector_account_category", "sector"], default="global")
    threshold_n = np.select([use_group, use_sector], [temp["group_n"], temp["sector_n"]], default=global_n)

    valid_threshold = eligible.to_numpy() & pd.notna(low) & pd.notna(high) & (low < high)
    bands = np.full(len(result), pd.NA, dtype=object)
    bands[valid_threshold & (values.to_numpy() <= low)] = "Low"
    bands[valid_threshold & (values.to_numpy() > low) & (values.to_numpy() < high)] = "Medium"
    bands[valid_threshold & (values.to_numpy() >= high)] = "High"

    result[f"{prefix}_eligible"] = eligible.astype(bool)
    result[f"{prefix}_band"] = pd.Series(bands, index=result.index, dtype="string")
    result[f"{prefix}_threshold_scope"] = pd.Series(np.where(eligible, scope, pd.NA), index=result.index, dtype="string")
    result[f"{prefix}_threshold_n"] = pd.Series(np.where(eligible, threshold_n, np.nan), index=result.index).astype("Int64")
    result[f"{prefix}_low_threshold"] = np.where(eligible, low, np.nan)
    result[f"{prefix}_high_threshold"] = np.where(eligible, high, np.nan)
    return result


## 5. Table 1: current financial status labels

These are current accounting-state descriptors and reason codes. They are not direct labels of financing demand.


In [ ]:
status = latest_all[[
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "period_end", "available_date", "financial_evidence_tier",
    "cash", "creditors_total", "current_assets", "debtors", "employees", "equity",
    "fixed_assets", "net_assets_liabilities", "net_current_assets_liabilities", "profit_loss",
    "total_assets_less_current_liabilities",
]].copy()
status = status.rename(columns={"period_end": "latest_period_end", "available_date": "latest_available_date"})

status["negative_equity_eligible"] = status["equity"].notna()
status["negative_equity_flag"] = (status["equity"] < 0).where(status["negative_equity_eligible"])
status["positive_equity_flag"] = (status["equity"] > 0).where(status["negative_equity_eligible"])

status["working_capital_deficit_eligible"] = status["net_current_assets_liabilities"].notna()
status["working_capital_deficit_flag"] = (status["net_current_assets_liabilities"] < 0).where(status["working_capital_deficit_eligible"])

status["reported_loss_eligible"] = status["profit_loss"].notna()
status["reported_loss_flag"] = (status["profit_loss"] < 0).where(status["reported_loss_eligible"])

status["creditors_cover_eligible"] = (
    status["current_assets"].notna() & status["creditors_total"].notna()
    & status["current_assets"].ge(0) & status["creditors_total"].ge(0)
)
status["creditors_exceed_current_assets_flag"] = (status["creditors_total"] > status["current_assets"]).where(status["creditors_cover_eligible"])
status["current_assets_cover_creditors_flag"] = (status["current_assets"] >= status["creditors_total"]).where(status["creditors_cover_eligible"])

status["cash_to_creditors_ratio"] = np.where(
    status["cash"].notna() & status["creditors_total"].gt(0),
    status["cash"] / status["creditors_total"], np.nan,
)
disclosed_assets = status["current_assets"].fillna(0) + status["fixed_assets"].fillna(0)
status["fixed_assets_to_disclosed_assets_ratio"] = np.where(
    status["fixed_assets"].notna() & status["current_assets"].notna() & disclosed_assets.gt(0),
    status["fixed_assets"] / disclosed_assets, np.nan,
)
status["debtors_to_current_assets_ratio"] = np.where(
    status["debtors"].notna() & status["current_assets"].gt(0),
    status["debtors"] / status["current_assets"], np.nan,
)
status["creditors_to_disclosed_assets_ratio"] = np.where(
    status["creditors_total"].notna() & disclosed_assets.gt(0),
    status["creditors_total"] / disclosed_assets, np.nan,
)
status["employees_per_million_disclosed_assets"] = np.where(
    status["employees"].ge(0) & disclosed_assets.gt(0),
    status["employees"] / (disclosed_assets / 1_000_000), np.nan,
)

for value_col, prefix in [
    ("cash_to_creditors_ratio", "cash_coverage"),
    ("fixed_assets_to_disclosed_assets_ratio", "asset_intensity"),
    ("debtors_to_current_assets_ratio", "receivables_intensity"),
    ("creditors_to_disclosed_assets_ratio", "creditor_intensity"),
    ("employees_per_million_disclosed_assets", "employee_intensity"),
]:
    status = add_quantile_band(status, value_col, prefix)

status.to_csv(OUTPUT_FILES["status"], index=False, encoding=CSV_ENCODING)
print("Table 1 rows:", f"{len(status):,}")


Table 1 rows: 100,000


## 6. Table 2: empirical financial scale labels

Each scale field receives its own Low/Medium/High band. No manually weighted composite business-opportunity score is created.


In [ ]:
scale = latest_all[[
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "period_end", "available_date", "financial_evidence_tier",
    "current_assets", "fixed_assets", "creditors_total", "equity", "employees",
    "net_assets_liabilities", "total_assets_less_current_liabilities",
]].copy()
scale = scale.rename(columns={"period_end": "latest_period_end", "available_date": "latest_available_date"})
scale["abs_equity_for_scale"] = scale["equity"].abs()

scale_inputs = {
    "current_assets": "current_assets_scale",
    "fixed_assets": "fixed_assets_scale",
    "creditors_total": "creditors_scale",
    "abs_equity_for_scale": "absolute_equity_scale",
    "employees": "employee_scale",
    "net_assets_liabilities": "net_assets_scale",
    "total_assets_less_current_liabilities": "total_assets_scale",
}
for value_col, prefix in scale_inputs.items():
    transformed = f"_{prefix}_signed_log"
    scale[transformed] = signed_log1p(scale[value_col])
    eligible = scale[value_col].notna()
    if value_col in {"current_assets", "fixed_assets", "creditors_total", "abs_equity_for_scale", "employees"}:
        eligible &= scale[value_col].ge(0)
    scale = add_quantile_band(scale, transformed, prefix, eligibility=eligible)

scale["financial_scale_available_field_count"] = scale[[f"{prefix}_eligible" for prefix in scale_inputs.values()]].sum(axis=1)
scale = scale.drop(columns=[c for c in scale.columns if c.startswith("_")])
scale.to_csv(OUTPUT_FILES["scale"], index=False, encoding=CSV_ENCODING)
print("Table 2 rows:", f"{len(scale):,}")


Table 2 rows: 100,000


## 7. Build adjacent annual company-period pairs

A valid pair has a gap of 250–550 days. Change values describe the movement from period `t` to period `t+1`; they must not be used as features at time `t`.


In [ ]:
panel = panel.sort_values(["CompanyNumber_norm", "period_end"]).reset_index(drop=True)
grouped = panel.groupby("CompanyNumber_norm", sort=False)
panel["period_t_plus_1"] = grouped["period_end"].shift(-1)
panel["available_date_t_plus_1"] = grouped["available_date"].shift(-1)
panel["evidence_tier_t_plus_1"] = grouped["financial_evidence_tier"].shift(-1)
panel["gap_days"] = (panel["period_t_plus_1"] - panel["period_end"]).dt.days

PAIR_METRICS = [
    "current_assets", "fixed_assets", "creditors_total", "equity", "net_assets_liabilities",
    "net_current_assets_liabilities", "cash", "debtors", "employees", "profit_loss",
    "total_assets_less_current_liabilities",
]
for metric in PAIR_METRICS:
    panel[f"{metric}_t_plus_1"] = grouped[metric].shift(-1)

pairs = panel[
    panel["period_t_plus_1"].notna()
    & panel["gap_days"].between(ANNUAL_GAP_MIN_DAYS, ANNUAL_GAP_MAX_DAYS)
].copy()
pairs = pairs.rename(columns={
    "period_end": "period_t", "available_date": "available_date_t",
    "financial_evidence_tier": "evidence_tier_t",
})

print("Valid annual pairs:", f"{len(pairs):,}")
print("Companies with valid annual pairs:", f"{pairs['CompanyNumber_norm'].nunique():,}")


Valid annual pairs: 81,603
Companies with valid annual pairs: 76,611


## 8. Table 3: longitudinal financial change labels

Signed-log changes are used because accounting values are highly skewed and some fields can be negative. Low/Medium/High means relative decline/stability/growth within the selected comparison scope.


In [ ]:
change_base_cols = [
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1",
    "gap_days", "evidence_tier_t", "evidence_tier_t_plus_1",
]
change = pairs[change_base_cols].copy()

CHANGE_METRICS = [
    "current_assets", "fixed_assets", "creditors_total", "equity", "net_assets_liabilities",
    "net_current_assets_liabilities", "cash", "debtors", "employees", "profit_loss",
    "total_assets_less_current_liabilities",
]
nonnegative_metrics = {"current_assets", "fixed_assets", "creditors_total", "cash", "debtors", "employees"}

for metric in CHANGE_METRICS:
    value_t = pd.to_numeric(pairs[metric], errors="coerce")
    value_next = pd.to_numeric(pairs[f"{metric}_t_plus_1"], errors="coerce")
    eligible = value_t.notna() & value_next.notna()
    if metric in nonnegative_metrics:
        eligible &= value_t.ge(0) & value_next.ge(0)

    change[f"{metric}_t"] = value_t
    change[f"{metric}_t_plus_1"] = value_next
    change[f"{metric}_change_eligible"] = eligible
    change[f"{metric}_signed_log_change"] = (signed_log1p(value_next) - signed_log1p(value_t)).where(eligible)
    change[f"{metric}_percent_change"] = np.where(
        eligible & value_t.abs().gt(1e-12),
        (value_next - value_t) / value_t.abs(), np.nan,
    )
    change = add_quantile_band(change, f"{metric}_signed_log_change", f"{metric}_change", eligibility=eligible)

change["change_available_field_count"] = change[[f"{metric}_change_eligible" for metric in CHANGE_METRICS]].sum(axis=1)
change.to_csv(OUTPUT_FILES["change"], index=False, encoding=CSV_ENCODING)
print("Table 3 rows:", f"{len(change):,}")


Table 3 rows: 81,603


## 9. Table 4: financial state-transition labels

These are the strongest candidates for future-state prediction. Every target has a separate eligibility flag so missing facts are never treated as negative outcomes.


In [ ]:
transition = pairs[[
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1",
    "gap_days", "evidence_tier_t", "evidence_tier_t_plus_1",
]].copy()

equity_t = pairs["equity"]
equity_next = pairs["equity_t_plus_1"]
equity_eligible = equity_t.notna() & equity_next.notna()
transition["negative_equity_transition_eligible"] = equity_eligible
transition["negative_equity_onset_eligible"] = equity_eligible & equity_t.ge(0)
transition["negative_equity_recovery_eligible"] = equity_eligible & equity_t.lt(0)
transition["negative_equity_persistent_eligible"] = equity_eligible & equity_t.lt(0)
transition["negative_equity_onset_flag"] = equity_next.lt(0).where(transition["negative_equity_onset_eligible"])
transition["negative_equity_recovery_flag"] = equity_next.ge(0).where(transition["negative_equity_recovery_eligible"])
transition["negative_equity_persistent_flag"] = equity_next.lt(0).where(transition["negative_equity_persistent_eligible"])

working_t = pairs["net_current_assets_liabilities"]
working_next = pairs["net_current_assets_liabilities_t_plus_1"]
working_eligible = working_t.notna() & working_next.notna()
transition["working_capital_transition_eligible"] = working_eligible
transition["working_capital_deficit_onset_eligible"] = working_eligible & working_t.ge(0)
transition["working_capital_deficit_recovery_eligible"] = working_eligible & working_t.lt(0)
transition["working_capital_deficit_persistent_eligible"] = working_eligible & working_t.lt(0)
transition["working_capital_deficit_onset_flag"] = working_next.lt(0).where(transition["working_capital_deficit_onset_eligible"])
transition["working_capital_deficit_recovery_flag"] = working_next.ge(0).where(transition["working_capital_deficit_recovery_eligible"])
transition["working_capital_deficit_persistent_flag"] = working_next.lt(0).where(transition["working_capital_deficit_persistent_eligible"])

creditors_t = pairs["creditors_total"]
creditors_next = pairs["creditors_total_t_plus_1"]
assets_t = pairs["current_assets"]
assets_next = pairs["current_assets_t_plus_1"]
creditor_eligible = creditors_t.notna() & creditors_next.notna() & assets_t.notna() & assets_next.notna()
pressure_t = creditors_t > assets_t
pressure_next = creditors_next > assets_next
transition["creditor_pressure_transition_eligible"] = creditor_eligible
transition["creditor_pressure_onset_eligible"] = creditor_eligible & (~pressure_t)
transition["creditor_pressure_recovery_eligible"] = creditor_eligible & pressure_t
transition["creditor_pressure_persistent_eligible"] = creditor_eligible & pressure_t
transition["creditor_pressure_onset_flag"] = pressure_next.where(transition["creditor_pressure_onset_eligible"])
transition["creditor_pressure_recovery_flag"] = (~pressure_next).where(transition["creditor_pressure_recovery_eligible"])
transition["creditor_pressure_persistent_flag"] = pressure_next.where(transition["creditor_pressure_persistent_eligible"])

profit_t = pairs["profit_loss"]
profit_next = pairs["profit_loss_t_plus_1"]
profit_eligible = profit_t.notna() & profit_next.notna()
transition["reported_loss_transition_eligible"] = profit_eligible
transition["reported_loss_onset_eligible"] = profit_eligible & profit_t.ge(0)
transition["reported_loss_recovery_eligible"] = profit_eligible & profit_t.lt(0)
transition["reported_loss_onset_flag"] = profit_next.lt(0).where(transition["reported_loss_onset_eligible"])
transition["reported_loss_recovery_flag"] = profit_next.ge(0).where(transition["reported_loss_recovery_eligible"])

transition.to_csv(OUTPUT_FILES["transition"], index=False, encoding=CSV_ENCODING)
print("Table 4 rows:", f"{len(transition):,}")


Table 4 rows: 81,603


## 10. Table 5: financial data-quality and evidence labels

These fields are controls for filtering, confidence, sample weighting, and sensitivity analysis. They are not business-opportunity outcomes.


In [ ]:
panel["useful_financial_period_flag"] = panel["financial_evidence_tier"].isin([
    "T1_observed_turnover", "T2_balance_sheet_rich", "T3_balance_sheet_partial",
])
filing_agg = panel.groupby("CompanyNumber_norm").agg(
    matched_account_periods=("period_end", "nunique"),
    useful_financial_periods=("useful_financial_period_flag", "sum"),
    first_period_end=("period_end", "min"),
    latest_period_end=("period_end", "max"),
    has_any_turnover=("turnover", lambda x: x.notna().any()),
    turnover_periods=("turnover", lambda x: int(x.notna().sum())),
).reset_index()

panel_quality = panel.sort_values(["CompanyNumber_norm", "period_end"]).copy()
panel_quality["previous_evidence_rank"] = panel_quality.groupby("CompanyNumber_norm")["evidence_rank"].shift(1)
panel_quality["previous_period_end"] = panel_quality.groupby("CompanyNumber_norm")["period_end"].shift(1)
panel_quality["previous_gap_days"] = (panel_quality["period_end"] - panel_quality["previous_period_end"]).dt.days
latest_quality = panel_quality.groupby("CompanyNumber_norm", as_index=False).tail(1)[[
    "CompanyNumber_norm", "previous_evidence_rank", "previous_gap_days",
]]

quality = latest_all[[
    "CompanyNumber_norm", "CompanyName", "primary_sector", "Accounts_AccountCategory",
    "period_end", "available_date", "financial_evidence_tier", "evidence_rank",
    "available_core_proxy_field_count", "available_any_financial_field_count",
    *[metric for metric in FINANCIAL_METRICS if metric in latest_all.columns],
]].copy()
quality = quality.rename(columns={"period_end": "latest_period_end", "available_date": "latest_available_date"})
quality = quality.merge(filing_agg, on="CompanyNumber_norm", how="left", validate="one_to_one", suffixes=("", "_agg"))
quality = quality.merge(latest_quality, on="CompanyNumber_norm", how="left", validate="one_to_one")

quality["has_matched_accounts_flag"] = quality["matched_account_periods"].fillna(0).ge(1)
quality["has_two_plus_financial_periods_flag"] = quality["useful_financial_periods"].fillna(0).ge(2)
quality["has_two_plus_turnover_periods_flag"] = quality["turnover_periods"].fillna(0).ge(2)
quality["has_any_turnover_flag"] = quality["has_any_turnover"].fillna(False).astype(bool)
quality["useful_financial_evidence_flag"] = quality["financial_evidence_tier"].isin([
    "T1_observed_turnover", "T2_balance_sheet_rich", "T3_balance_sheet_partial",
])
quality["core_fields_complete_flag"] = quality["available_core_proxy_field_count"].fillna(0).ge(len(CORE_PROXY_FIELDS))
quality["accounts_age_days_at_snapshot"] = (SNAPSHOT_DATE - quality["latest_period_end"]).dt.days
quality["accounts_older_than_24m_flag"] = quality["accounts_age_days_at_snapshot"].gt(730).where(quality["latest_period_end"].notna())

nonnegative_fields = ["cash", "current_assets", "fixed_assets", "debtors", "employees"]
negative_checks = pd.DataFrame({field: quality[field].lt(0) for field in nonnegative_fields})
quality["impossible_negative_value_flag"] = negative_checks.any(axis=1)
quality["impossible_negative_fields"] = negative_checks.apply(
    lambda row: "|".join(row.index[row].tolist()), axis=1,
)

amount_fields = ["cash", "creditors_total", "current_assets", "debtors", "equity", "fixed_assets", "net_assets_liabilities", "total_assets_less_current_liabilities"]
outlier_flags = []
outlier_thresholds = {}
for field in amount_fields:
    absolute = quality[field].abs()
    threshold = absolute.quantile(0.999)
    outlier_thresholds[field] = threshold
    outlier_flags.append(absolute.gt(threshold) & absolute.notna())
quality["extreme_amount_p999_flag"] = pd.concat(outlier_flags, axis=1).any(axis=1)
quality["latest_period_gap_anomaly_flag"] = (~quality["previous_gap_days"].between(ANNUAL_GAP_MIN_DAYS, ANNUAL_GAP_MAX_DAYS)).where(quality["previous_gap_days"].notna())
quality["evidence_improved_flag"] = (quality["evidence_rank"] < quality["previous_evidence_rank"]).where(quality["previous_evidence_rank"].notna())
quality["evidence_deteriorated_flag"] = (quality["evidence_rank"] > quality["previous_evidence_rank"]).where(quality["previous_evidence_rank"].notna())
quality["extreme_amount_thresholds_json"] = json.dumps({k: None if pd.isna(v) else float(v) for k, v in outlier_thresholds.items()})

drop_quality_cols = [*FINANCIAL_METRICS, "evidence_rank", "previous_evidence_rank", "has_any_turnover"]
quality = quality.drop(columns=[c for c in drop_quality_cols if c in quality.columns])
quality = quality.drop(columns=["latest_period_end_agg"], errors="ignore")
quality.to_csv(OUTPUT_FILES["quality"], index=False, encoding=CSV_ENCODING)
print("Table 5 rows:", f"{len(quality):,}")


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_11008\3784919473.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quality["has_any_turnover_flag"] = quality["has_any_turnover"].fillna(False).astype(bool)


Table 5 rows: 100,000


## 11. Generate bilingual Markdown guides

The guides include observed row counts and positive-label distributions from this run.


In [ ]:
def flag_line(df, flag, eligible=None):
    if eligible and eligible in df:
        valid = df[eligible].fillna(False)
    else:
        valid = df[flag].notna()
    positive = df.loc[valid, flag].fillna(False).astype(bool)
    n = int(valid.sum())
    p = int(positive.sum())
    return p, n, (p / n if n else np.nan)


neg_eq = flag_line(status, "negative_equity_flag", "negative_equity_eligible")
wc_def = flag_line(status, "working_capital_deficit_flag", "working_capital_deficit_eligible")
loss = flag_line(status, "reported_loss_flag", "reported_loss_eligible")
neg_onset = flag_line(transition, "negative_equity_onset_flag", "negative_equity_onset_eligible")
wc_onset = flag_line(transition, "working_capital_deficit_onset_flag", "working_capital_deficit_onset_eligible")

guide_cn = f"""# 财务数据分析五大表使用说明

## 数据范围

- 公司母表：`UKcompanies_active_account_category_sample_100k.csv`，共 {len(companies):,} 家公司。
- 财务来源：2024年7月至2026年6月的24个Companies House Accounts Monthly Bulk ZIP。
- 当前快照：每家公司最新的account period，而不是历史best-evidence period。
- 分位标签：优先采用`primary_sector + Accounts_AccountCategory`组内30%/70%分位数；有效样本少于{MIN_GROUP_N}或阈值相同时，回退到行业，再回退到总体。
- 原始公司表和Accounts ZIP不会被修改。

## 文件与作用

### 1. `01_financial_status_labels_100k.csv`

一家公司一行，共 {len(status):,} 行。描述最新期间的财务状态和reason codes，包括负权益、营运资金缺口、reported loss、债权人压力，以及现金覆盖、资产密集度、应收账款密集度等分位标签。

- `negative_equity_flag`：{neg_eq[0]:,}/{neg_eq[1]:,}，占有效样本 {neg_eq[2]:.2%}。
- `working_capital_deficit_flag`：{wc_def[0]:,}/{wc_def[1]:,}，占有效样本 {wc_def[2]:.2%}。
- `reported_loss_flag`：{loss[0]:,}/{loss[1]:,}，占有效样本 {loss[2]:.2%}；因覆盖率低，只建议作为辅助信号。

用途：当前公司画像、模型输入、结果解释。它们不是融资需求的直接标签。

### 2. `02_financial_scale_labels_100k.csv`

一家公司一行，共 {len(scale):,} 行。分别为current assets、fixed assets、creditors、absolute equity、employees、net assets和total assets生成Low/Medium/High规模带。

用途：三分类模型特征、行业内规模比较、缺少turnover公司的规模proxy。该表不构造人工加权的商业机会总分，也不能替代Lloyds正式BB/SME/Mid Corporate层级。

### 3. `03_financial_change_labels.csv`

一条相邻年度company-period pair一行，共 {len(change):,} 行，涉及 {change['CompanyNumber_norm'].nunique():,} 家公司。包含原值、下一期值、signed-log change、percent change和30/70变化等级。

用途：描述增长、稳定或收缩；可以把历史变化作为下一次预测的输入，也可以把下一期变化作为target。`t -> t+1`变化在时间`t`尚不可见，不能当作时间`t`特征。

### 4. `04_financial_transition_labels.csv`

一条相邻年度company-period pair一行，共 {len(transition):,} 行。记录负权益、营运资金缺口、债权人压力和reported loss的发生、持续与恢复。

- `negative_equity_onset_flag`：{neg_onset[0]:,}/{neg_onset[1]:,}，占有效pair {neg_onset[2]:.2%}。
- `working_capital_deficit_onset_flag`：{wc_onset[0]:,}/{wc_onset[1]:,}，占有效pair {wc_onset[2]:.2%}。

用途：训练下一期财务状态模型。每个target必须与相应`*_eligible`字段一起使用，缺失不能当作False。

### 5. `05_financial_data_quality_labels_100k.csv`

一家公司一行，共 {len(quality):,} 行。包括T1-T4 evidence tier、期间数量、turnover覆盖、字段完整度、账目时效、异常负值、极端金额和证据等级变化。

用途：过滤、样本加权、置信度、敏感性分析和数据质量监控。它不是业务机会标签。

## 连接键

- 公司级表：使用`CompanyNumber_norm`连接。
- 跨期表：使用`CompanyNumber_norm + period_t + period_t_plus_1`唯一定位pair。
- 连接多维度数据时必须保证news和hiring变量在相应`available_date_t`之前已经可获得。

## Feature、Target与Control

- 当前状态及规模标签通常是feature或reason code。
- `*_onset_flag`、`*_recovery_flag`及下一期变化等级可以作为未来target。
- T级、完整度和异常标记是control/confidence字段。
- 三分类的正式target仍应由observed turnover生成：BB < £3m，SME £3m–£25m，Mid Corporate £25m–£500m。

## 重要限制

- Accounts数据主要是年度披露，下一期财务状态不等于未来4–6个月融资需求。
- `creditors_total`不一定等于严格会计定义的current liabilities，相关ratio应称为proxy。
- 分位标签是样本相对位置，不是通用财务健康标准。
- 未来模型应按照`available_date`做时间切分，不能随机把未来filing放入训练集。
"""

guide_en = f"""# Guide to the Five Financial Analysis Tables

## Data scope

- Company universe: `UKcompanies_active_account_category_sample_100k.csv`, containing {len(companies):,} companies.
- Financial source: 24 Companies House Accounts Monthly Bulk ZIPs from July 2024 to June 2026.
- Current snapshot: each company's latest account period, not its historical best-evidence period.
- Quantile labels: 30th/70th percentiles within `primary_sector + Accounts_AccountCategory`; groups with fewer than {MIN_GROUP_N} eligible observations or tied thresholds fall back to sector and then global thresholds.
- The raw company list and Accounts ZIP files are never modified.

## Files and roles

### 1. `01_financial_status_labels_100k.csv`

One row per company, {len(status):,} rows. It describes the latest accounting state and reason codes: negative equity, working-capital deficit, reported loss, creditor pressure, cash coverage, asset intensity, receivables intensity and related empirical bands.

- `negative_equity_flag`: {neg_eq[0]:,}/{neg_eq[1]:,}, or {neg_eq[2]:.2%} of eligible companies.
- `working_capital_deficit_flag`: {wc_def[0]:,}/{wc_def[1]:,}, or {wc_def[2]:.2%} of eligible companies.
- `reported_loss_flag`: {loss[0]:,}/{loss[1]:,}, or {loss[2]:.2%} of eligible companies; use only as a secondary signal because coverage is sparse.

Use: current company profiling, model features and explanations. These fields are not direct financing-demand labels.

### 2. `02_financial_scale_labels_100k.csv`

One row per company, {len(scale):,} rows. It provides separate Low/Medium/High bands for current assets, fixed assets, creditors, absolute equity, employees, net assets and total assets.

Use: three-class model features, within-sector scale comparison and scale proxies where turnover is unavailable. It does not create a manually weighted commercial-opportunity score and does not replace Lloyds' formal BB/SME/Mid Corporate segmentation.

### 3. `03_financial_change_labels.csv`

One row per adjacent annual company-period pair, {len(change):,} rows across {change['CompanyNumber_norm'].nunique():,} companies. It contains values at `t` and `t+1`, signed-log changes, percentage changes and empirical 30/70 change bands.

Use: describing growth, stability or contraction. Historical changes can be model features, while next-period changes can be prediction targets. A `t -> t+1` change is not known at time `t` and must not be used as a time-`t` feature.

### 4. `04_financial_transition_labels.csv`

One row per adjacent annual company-period pair, {len(transition):,} rows. It records onset, persistence and recovery for negative equity, working-capital deficit, creditor pressure and reported loss.

- `negative_equity_onset_flag`: {neg_onset[0]:,}/{neg_onset[1]:,}, or {neg_onset[2]:.2%} of eligible pairs.
- `working_capital_deficit_onset_flag`: {wc_onset[0]:,}/{wc_onset[1]:,}, or {wc_onset[2]:.2%} of eligible pairs.

Use: next-period financial-state prediction. Every target must be filtered by its corresponding `*_eligible` field; missing observations must never be converted to False.

### 5. `05_financial_data_quality_labels_100k.csv`

One row per company, {len(quality):,} rows. It contains T1-T4 evidence tier, period counts, turnover coverage, field completeness, account recency, impossible negative values, extreme amounts and evidence-tier movements.

Use: filtering, sample weighting, confidence, sensitivity analysis and data-quality monitoring. It is not a commercial-opportunity outcome.

## Join keys

- Company-level tables: join on `CompanyNumber_norm`.
- Longitudinal tables: uniquely identify a pair with `CompanyNumber_norm + period_t + period_t_plus_1`.
- When joining news or hiring data, ensure that every variable was observable on or before `available_date_t`.

## Feature, target and control roles

- Current status and scale labels are usually features or reason codes.
- `*_onset_flag`, `*_recovery_flag` and next-period change bands can be future targets.
- Evidence tier, completeness and anomaly fields are controls/confidence variables.
- The formal three-class target should still be generated from observed turnover: BB below £3m, SME £3m–£25m, and Mid Corporate £25m–£500m.

## Important limitations

- Accounts are mainly annual disclosures; a next-account financial state is not the same as financing demand in the next four to six months.
- `creditors_total` is not necessarily identical to strictly defined current liabilities, so related ratios are proxies.
- Quantile labels are relative positions within this sample, not universal financial-health standards.
- Future models must use `available_date` for temporal splitting and must not place future filings in the training set.
"""

OUTPUT_FILES["guide_cn"].write_text(guide_cn, encoding="utf-8")
OUTPUT_FILES["guide_en"].write_text(guide_en, encoding="utf-8")
print("Bilingual guides written")


Bilingual guides written


C:\Users\RoHenry\AppData\Local\Temp\ipykernel_11008\258859121.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  positive = df.loc[valid, flag].fillna(False).astype(bool)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_11008\258859121.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  positive = df.loc[valid, flag].fillna(False).astype(bool)
C:\Users\RoHenry\AppData\Local\Temp\ipykernel_11008\258859121.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objec

## 12. Final quality assurance


In [ ]:
for name, frame in [("status", status), ("scale", scale), ("quality", quality)]:
    assert len(frame) == 100_000, f"{name} must contain exactly 100,000 rows"
    assert frame["CompanyNumber_norm"].nunique() == 100_000, f"{name} company key is not unique"

for name, frame in [("change", change), ("transition", transition)]:
    pair_keys = ["CompanyNumber_norm", "period_t", "period_t_plus_1"]
    assert not frame.duplicated(pair_keys).any(), f"Duplicate company-period pairs in {name}"
    assert frame["gap_days"].between(ANNUAL_GAP_MIN_DAYS, ANNUAL_GAP_MAX_DAYS).all(), f"Invalid gap in {name}"

for path in OUTPUT_FILES.values():
    assert path.exists() and path.stat().st_size > 0, f"Missing or blank output: {path}"

output_summary = pd.DataFrame([
    {"file": OUTPUT_FILES["status"].name, "rows": len(status), "grain": "company"},
    {"file": OUTPUT_FILES["scale"].name, "rows": len(scale), "grain": "company"},
    {"file": OUTPUT_FILES["change"].name, "rows": len(change), "grain": "company-period pair"},
    {"file": OUTPUT_FILES["transition"].name, "rows": len(transition), "grain": "company-period pair"},
    {"file": OUTPUT_FILES["quality"].name, "rows": len(quality), "grain": "company"},
])
display(output_summary)
print("All quality checks passed.")


,file,rows,grain
0,01_financial_status_labels_100k.csv,100000,company
1,02_financial_scale_labels_100k.csv,100000,company
2,03_financial_change_labels.csv,81603,company-period pair
3,04_financial_transition_labels.csv,81603,company-period pair
4,05_financial_data_quality_labels_100k.csv,100000,company


All quality checks passed.
